# Vibration Structural Analysis & Signal Processing Recipe

This recipe combines 3 `algebrax` tools to evaluate mechanical structural dynamics:

1. **Permutation Group Symmetries** (`algebrax.group.compose` & `signature`):
   Models rotational and reflectional symmetries ($r, s \in D_4$) over multi-rotor turbine systems.
2. **Stiffness Matrix Determinants** (`algebrax.matrix.academic.determinant`):
   Computes matrix determinants $\det(K)$ to verify structural rigidity and stability.
3. **Instantaneous Amplitude Envelopes** (`algebrax.transforms.hilbert`):
   Derives analytic signals $a[n] = x[n] + j \mathcal{H}\{x[n]\}$ to extract instantaneous envelopes $|a[n]|$ from vibration streams.

In [ ]:
import math

import algebrax as ax


## 1. Permutation Symmetries (Group Theory)

We compose 90° rotation ($r$) and horizontal flip ($s$) permutations over 4 turbine nodes.

In [ ]:

rot_90 = {0: 1, 1: 2, 2: 3, 3: 0}
flip_h = {0: 1, 1: 0, 2: 3, 3: 2}

combined = ax.group.compose(rot_90, flip_h)

print(f'Rotation Permutation (r):    {rot_90} [Parity: {ax.group.signature(rot_90):+d}]')
print(f'Reflection Permutation (s):  {flip_h} [Parity: {ax.group.signature(flip_h):+d}]')
print(f'Combined Motion (s o r):     {combined} [Parity: {ax.group.signature(combined):+d}]')

## 2. Structural Stiffness Matrix Determinant

We compute $\det(K)$ to audit structural stability.

In [ ]:
import algebrax as ax

stiffness_matrix = {
    0: {0: 4.0, 1: -2.0, 2: 0.0},
    1: {0: -2.0, 1: 5.0, 2: -3.0},
    2: {0: 0.0, 1: -3.0, 2: 4.0},
}

det_k = ax.matrix.academic.determinant(stiffness_matrix)
print(f'Stiffness Matrix Determinant det(K): {det_k:.2f}')

## 3. Instantaneous Hilbert Envelope Extraction

We compute the Hilbert transform $\mathcal{H}\{x[n]\}$ and extract the amplitude envelope $|a[n]| = \sqrt{x[n]^2 + \mathcal{H}\{x[n]\}^2}$.

In [ ]:
import algebrax as ax

n_samples = 16
raw_vibration = {i: math.sin(2 * math.pi * i / 4) * (3.0 if 6 <= i <= 10 else 1.0) for i in range(n_samples)}
analytic_signal = ax.transforms.hilbert(raw_vibration, n=n_samples)

print('Index | Raw Signal | Envelope |a[n]|')
for i in range(n_samples):
    env_mag = abs(analytic_signal.get(i, 0j))
    print(f' {i:4d} | {raw_vibration[i]:10.4f} | {env_mag:14.4f}')